# Double Machine Learning and Heterogeneous Treatment Effects

*Companion notebook — Quasi-Experimental Methods for Business Interventions in Python. Self-contained: run the setup cells first.*

In [ ]:
# Install dependencies (Colab/Jupyter)
%pip install -q numpy pandas matplotlib seaborn statsmodels linearmodels scikit-learn scipy doubleml econml dowhy tfp-causalimpact pysyncon rdrobust

## Setup: shared imports and data generators

This cell reproduces the book's `_common.py` so the notebook runs standalone.

In [ ]:
"""Shared setup for all chapters: imports, plotting theme, and data generators."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

rng = np.random.default_rng(42)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")


def make_panel(
    n_stores=120,
    n_markets=12,
    n_weeks=104,                # 2 years of weekly data
    treat_markets=(0, 1, 2, 3), # which markets get the program
    treat_week=60,              # program launch week
    true_effect=8.0,            # true lift in weekly revenue ($k)
    effect_ramp=6,              # weeks for the effect to reach full size
    seed=42,
):
    """NorthShore Retail store-week panel with known treatment effect."""
    rng = np.random.default_rng(seed)
    stores = np.arange(n_stores)
    markets = stores % n_markets
    store_fe = rng.normal(100, 15, n_stores)
    market_fe = rng.normal(0, 8, n_markets)
    store_trend = rng.normal(0.05, 0.03, n_stores)

    rows = []
    market_shock = rng.normal(0, 3, (n_markets, n_weeks))
    for w in range(n_weeks):
        season = 6 * np.sin(2 * np.pi * w / 52)
        for s in stores:
            m = markets[s]
            treated_unit = m in treat_markets
            post = w >= treat_week
            if treated_unit and post:
                ramp = min(1.0, (w - treat_week + 1) / effect_ramp)
                tau = true_effect * ramp
            else:
                tau = 0.0
            y = (store_fe[s] + market_fe[m] + store_trend[s] * w
                 + season + market_shock[m, w]
                 + tau + rng.normal(0, 4))
            rows.append((s, m, w, int(treated_unit), int(post),
                         int(treated_unit and post), y))

    return pd.DataFrame(rows, columns=[
        "store", "market", "week", "treated", "post", "D", "revenue"])


def make_customers(n=8000, true_att=12.0, seed=42):
    """Loyalty-program adoption data with selection on observables."""
    rng = np.random.default_rng(seed)
    tenure = rng.gamma(3, 1.2, n)
    spend0 = rng.lognormal(4.0, 0.5, n)
    engage = rng.beta(2, 5, n) * 10
    urban = rng.binomial(1, 0.55, n)
    logit = -3.0 + 0.25*tenure + 0.004*spend0 + 0.30*engage + 0.4*urban
    p = 1 / (1 + np.exp(-logit))
    D = rng.binomial(1, p)
    y = (0.55*spend0 + 4*tenure + 2.5*engage + 6*urban
         + true_att * D + rng.normal(0, 12, n))
    return pd.DataFrame(dict(tenure=tenure, spend0=spend0, engage=engage,
                             urban=urban, D=D, y=y))


panel = make_panel()
cust = make_customers()
X_cols = ["tenure", "spend0", "engage", "urban"]
TRUE_ATT = 12.0


Modern business datasets have **hundreds of covariates** (RFM features, behavioral telemetry, demographics). Plugging ML predictions naively into a causal regression produces **regularization bias**; Double/Debiased Machine Learning (Chernozhukov et al. 2018) fixes this with two ideas:

1. **Neyman orthogonality:** estimate the effect from *residualized* outcome and treatment (a partialling-out moment condition insensitive to small nuisance errors).
2. **Cross-fitting:** fit nuisance models on one fold, evaluate residuals on another, so overfitting doesn't leak into the effect estimate.

The identifying assumption is still **conditional ignorability** — DML does not solve unobserved confounding; it just lets you adjust for observed confounders flexibly and honestly.

## Average effect with `DoubleML`

Reusing the loyalty-program data from Section 9, now with gradient boosting for both nuisances:

In [ ]:
from doubleml import DoubleMLData, DoubleMLPLR
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier

dml_data = DoubleMLData(cust, y_col="y", d_cols="D", x_cols=X_cols)
plr = DoubleMLPLR(dml_data,
                  ml_l=GradientBoostingRegressor(),   # E[Y|X]
                  ml_m=GradientBoostingClassifier(),  # E[D|X]
                  n_folds=5, n_rep=2)
plr.fit()
print(plr.summary)        # coef ≈ effect, with valid SEs

Notes: the PLR (partially linear) model assumes a constant effect; the IRM model (`DoubleMLIRM`, Section 9.6) allows full interaction and targets ATE/ATT directly. Always sanity-check the nuisance learners' out-of-fold predictive performance (`plr.evaluate_learners()`); garbage nuisances → noisy effects.

## Heterogeneous effects (CATE) with `econml`: who benefits most?

For business, the question after "did it work?" is "**for whom** did it work?" — that drives targeting and ROI. Suppose the loyalty effect actually grows with engagement:

In [ ]:
TRUE_CATE = 4 + 2.2 * cust.engage          # effect increases with engagement
cust["y_het"] = (0.55*cust.spend0 + 4*cust.tenure + 2.5*cust.engage
                 + 6*cust.urban + TRUE_CATE*cust.D
                 + rng.normal(0, 12, len(cust)))

In [ ]:
from econml.dml import CausalForestDML
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

X = cust[["tenure", "spend0", "engage", "urban"]].values
cf = CausalForestDML(
    model_y=RandomForestRegressor(min_samples_leaf=20),
    model_t=RandomForestClassifier(min_samples_leaf=20),
    discrete_treatment=True, n_estimators=800, random_state=0)
cf.fit(Y=cust.y_het.values, T=cust.D.values, X=X)

cust["cate"] = cf.effect(X)
lo, hi = cf.effect_interval(X, alpha=0.05)

plt.figure(figsize=(8, 4))
plt.scatter(cust.engage, cust.cate, s=4, alpha=0.3)
plt.plot(np.sort(cust.engage), 4 + 2.2*np.sort(cust.engage),
         "r--", label="true CATE")
plt.xlabel("Engagement"); plt.ylabel("Estimated CATE ($/month)")
plt.legend(); plt.tight_layout(); plt.show()

## From CATEs to decisions: uplift-style evaluation

Rank customers by estimated CATE and ask whether targeting the top deciles concentrates the gains — a **qini/uplift curve** logic:

In [ ]:
cust["decile"] = pd.qcut(cust.cate, 10, labels=False)
by_dec = (cust.groupby(["decile", "D"]).y_het.mean().unstack("D"))
by_dec["uplift"] = by_dec[1] - by_dec[0]
by_dec["uplift"].plot(kind="bar", figsize=(8, 3.5),
                      ylabel="Observed uplift ($)", xlabel="CATE decile")
plt.tight_layout(); plt.show()

A monotone pattern validates the CATE model *directionally*. For formal validation, hold out data and use doubly robust scores (e.g., `econml`'s `DRTester` or qini coefficients on a holdout). The end product — "target segments with CATE > cost" — is where causal inference pays its salary.

## DoWhy refutation harness

`dowhy` wraps any estimate in automated falsification tests — cheap insurance worth running on every observational analysis:

In [ ]:
model = CausalModel(data=cust, treatment="D", outcome="y",
                    common_causes=X_cols)
estimand = model.identify_effect(proceed_when_unidentifiable=True)
estimate = model.estimate_effect(
    estimand, method_name="backdoor.propensity_score_weighting")

for refuter in ["random_common_cause",       # add noise covariate: effect should not move
                "placebo_treatment_refuter", # shuffle D: effect should vanish
                "data_subset_refuter"]:      # subsample: effect should be stable
    print(model.refute_estimate(estimand, estimate, method_name=refuter))